In [ ]:
!pip install -q langchain langchain-google-genai redis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.4/502.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 27.4 MB/s eta 0:00:00


In [ ]:
import os
import json

from google.colab import userdata
from redis import Redis

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    message_to_dict,
    messages_from_dict
)

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

REDIS_URL = userdata.get("REDIS_URL")

redis_client = Redis.from_url(
    REDIS_URL,
    decode_responses=True
)

redis_client.ping()

True

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [ ]:
human_message = HumanMessage(
    content="My name is Sameer."
)

ai_message = AIMessage(
    content="Hello Sameer."
)

In [ ]:
human_message

HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={})

In [ ]:
ai_message

AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])

In [ ]:
human_json = json.dumps(message_to_dict(human_message))

In [ ]:
ai_json = json.dumps(message_to_dict(ai_message))

### Sending to Redis

In [ ]:
THREAD_ID = "thread_101"
MESSAGES_KEY = f"chat:{THREAD_ID}:messages"
redis_client.delete(MESSAGES_KEY)

0

In [ ]:
redis_client.lpush(MESSAGES_KEY, human_json)

1

In [ ]:
redis_client.lpush(MESSAGES_KEY, ai_json)

2

In [ ]:
stored_json = redis_client.lrange(
    MESSAGES_KEY,
    0,
    -1
)

stored_json

['{"type": "ai", "data": {"content": "Hello Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "ai", "name": null, "id": null, "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": null}}',
 '{"type": "human", "data": {"content": "My name is Sameer.", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null}}']

In [ ]:
lst = []
lst.append(2)
lst.append(3)
lst

[2, 3]

In [ ]:
[3,2]

[3, 2]

In [ ]:
lst = redis_client.lrange(MESSAGES_KEY, 0,-1)

In [ ]:
stored_dicts = [
    json.loads(item)
    for item in lst
]

In [ ]:
messages_from_dict(stored_dicts)

[AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={})]

In [ ]:
def load_history(thread_id):

    key = f"chat:{thread_id}:messages"

    stored_messages = redis_client.lrange(
        key,
        0,
        -1
    )

    # LPUSH returns newest message first.
    stored_messages.reverse()

    message_dicts = [
        json.loads(message)
        for message in stored_messages
    ]

    history = InMemoryChatMessageHistory()

    history.add_messages(
        messages_from_dict(message_dicts)
    )

    return history

In [ ]:
def save_message(thread_id, message):

    key = f"chat:{thread_id}:messages"

    redis_client.lpush(
        key,
        json.dumps(
            message_to_dict(message)
        )
    )

In [ ]:
human_message = HumanMessage(content = "Hi I'm good what about you?")
save_message(THREAD_ID, human_message)

In [ ]:
load_history(THREAD_ID)

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Hi I'm good what about you?", additional_kwargs={}, response_metadata={})])

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are ERICA, a helpful banking assistant. "
        "Answer briefly and remember the conversation."
    ),

    MessagesPlaceholder(
        variable_name="history"
    ),

    (
        "human",
        "{user_input}"
    )
])

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [ ]:
chain = prompt | llm

In [ ]:
def chat(thread_id, user_input):

    # Load previous conversation from Redis.
    history = load_history(thread_id)

    # Send previous history and current input to the LLM.
    response = chain.invoke({
        "history": history.messages,
        "user_input": user_input
    })

    # Save the current turn in Redis.
    save_message(
        thread_id,
        HumanMessage(content=user_input)
    )

    save_message(
        thread_id,
        response
    )

    return response.content

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Chat ended.")
        break

    response = chat(
        THREAD_ID,
        user_input
    )

    print("ERICA:", response)

You: Hi Erica
ERICA: Hello Sameer, I'm ready to assist you. How can I help today?
You: How are you doing?
ERICA: As an AI, I don't experience "doing" in the way humans do, but I am fully operational and ready to assist you, Sameer. How can I help you today?
You: exit
Chat ended.


In [ ]:
load_history(THREAD_ID)

InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Sameer.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Sameer.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Hi I'm good what about you?", additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi Erica', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Sameer, I'm ready to assist you. How can I help today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8fd8-b1a7-78d3-ac87-702707eaf3e9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 380, 'total_tokens': 422, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 362}}), HumanMessage(content='How are you doing?', additional_kwargs={}, response_metadata={}

In [ ]:
redis_client.lrange(MESSAGES_KEY, 0,-1)

['{"type": "ai", "data": {"content": "As an AI, I don\'t experience \\"doing\\" in the way humans do, but I am fully operational and ready to assist you, Sameer. How can I help you today?", "additional_kwargs": {}, "response_metadata": {"finish_reason": "STOP", "model_name": "gemini-2.5-flash", "safety_ratings": [], "model_provider": "google_genai"}, "type": "ai", "name": null, "id": "lc_run--019f8fd8-d23b-7633-9a09-3b3fe7c564ad-0", "tool_calls": [], "invalid_tool_calls": [], "usage_metadata": {"input_tokens": 67, "output_tokens": 104, "total_tokens": 171, "input_token_details": {"cache_read": 0}, "output_token_details": {"reasoning": 65}}}}',
 '{"type": "human", "data": {"content": "How are you doing?", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null}}',
 '{"type": "ai", "data": {"content": "Hello Sameer, I\'m ready to assist you. How can I help today?", "additional_kwargs": {}, "response_metadata": {"finish_reason": "STOP", "model_name": "g

## Summarization

In [ ]:
THREAD_ID = "customer_101"

RAW_MESSAGE_LIMIT = 10
RAW_CONTEXT_SIZE = 5

SUMMARY_CHUNK_SIZE = 5
SUMMARY_CONTEXT_SIZE = 5

In [ ]:
def save_message(thread_id, message):

    key = f"chat:{thread_id}:messages"

    redis_client.lpush(
        key,
        json.dumps(
            message_to_dict(message)
        )
    )

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Summarize this conversation chunk briefly.

Preserve:
- important user facts
- decisions
- requests
- unresolved work
"""
    ),

    (
        "human",
        "{conversation_chunk}"
    )
])

In [ ]:
summary_chain = summary_prompt|llm

In [ ]:
# Get message history key
# Make a summary history key


# Load initial 5 messages


# Convert them to InMemoryChatMEssagehistory

# Pass them to llm


# Delete the initial 5 messages?



In [ ]:
def summarize_history(thread_id):

    messages_key = f"chat:{thread_id}:messages"
    summaries_key = f"chat:{thread_id}:summaries"

    message_count = redis_client.llen(
        messages_key
    )

    if message_count < RAW_MESSAGE_LIMIT:
        return None

    # Fetch oldest five messages
    stored_chunk = redis_client.lrange(
        messages_key,
        -SUMMARY_CHUNK_SIZE,
        -1
    )

    # Convert them into oldest-first order
    stored_chunk.reverse()

    chunk_dicts = [
        json.loads(message)
        for message in stored_chunk
    ]

    chunk_messages = messages_from_dict(
        chunk_dicts
    )

    conversation_chunk = "\n".join(
        f"{message.type}: {message.content}"
        for message in chunk_messages
    )

    response = summary_chain.invoke({
        "conversation_chunk": conversation_chunk
    })

    # Store newest summary at index 0
    redis_client.lpush(
        summaries_key,
        response.content
    )

    # Delete the oldest five raw messages
    redis_client.ltrim(
        messages_key,
        0,
        -(SUMMARY_CHUNK_SIZE + 1)
    )

    return response.content

In [ ]:
from langchain_core.messages import HumanMessage
def load_history(thread_id):

    messages_key = f"chat:{thread_id}:messages"
    summaries_key = f"chat:{thread_id}:summaries"

    # Latest five summaries
    stored_summaries = redis_client.lrange(
        summaries_key,
        0,
        SUMMARY_CONTEXT_SIZE - 1
    )

    # Convert newest-first Redis order
    # into oldest-first summary order
    stored_summaries.reverse()

    # Latest five raw messages
    stored_messages = redis_client.lrange(
        messages_key,
        0,
        RAW_CONTEXT_SIZE - 1
    )

    # Convert newest-first Redis order
    # into oldest-first conversation order
    stored_messages.reverse()

    message_dicts = [
        json.loads(message)
        for message in stored_messages
    ]

    history = InMemoryChatMessageHistory()

    if stored_summaries:

        summary_text = "\n\n".join(
            f"Summary {index}: {summary}"
            for index, summary in enumerate(
                stored_summaries,
                start=1
            )
        )

        history.add_message(
            SystemMessage(
                content=(
                    "Earlier conversation summaries:\n\n"
                    f"{summary_text}"
                )
            )
        )

    history.add_messages(
        messages_from_dict(message_dicts)
    )

    return history

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are ERICA, a helpful banking assistant."
    ),

    MessagesPlaceholder("history"),

    (
        "human",
        "{user_input}"
    )
])
chat_chain = chat_prompt | llm

In [ ]:
def chat(thread_id, user_input):

    history = load_history(thread_id)

    response = chat_chain.invoke({
        "history": history.messages,
        "user_input": user_input
    })

    save_message(
        thread_id,
        HumanMessage(content=user_input)
    )

    save_message(
        thread_id,
        response
    )

    new_summary = summarize_history(
        thread_id
    )

    if new_summary:

        print(
            "\n[Memory] Oldest 5 messages "
            "summarized and deleted."
        )

    return response.content